# Laboratorio 4 — Análisis de Datos GeoEspaciales
## Notebook 1: Conexión a la API y descarga de datos (Ejercicios 1 y 2)

Lagos Atitlán y Amatitlán — detección de floraciones de cianobacteria con Sentinel-2 (Copernicus Data Space Ecosystem).

**Ejercicio 1.** Establecer conexión con la API de Sentinel-2 usando `openeo`.

**Ejercicio 2.** Obtener únicamente los datos raster necesarios para cada lago (bandas B03, B04, B08 para NDVI/NDWI vía openEO; resultado del script oficial de cianobacteria vía Sentinel Hub Process API), usando las fechas oficiales provistas en el enunciado.

### 0. Setup

Requiere las variables de entorno `SH_CLIENT_ID` y `SH_CLIENT_SECRET` (ver `.env.example` en la raíz del repo). Cópialas a un archivo `.env` local (NO lo subas a git, ya está en `.gitignore`) o expórtalas en tu shell antes de abrir Jupyter.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")

from src.config import LAGOS
from src import sentinel_api

### Ejercicio 1 — Conexión con la API de Sentinel-2 (openEO)

`get_openeo_connection()` intenta autenticarse por *client credentials* (silencioso, recomendado) si están definidas `SH_CLIENT_ID`/`SH_CLIENT_SECRET`; si no, cae a login interactivo por navegador (`authenticate_oidc()`), que debes completar manualmente.

In [ ]:
con = sentinel_api.get_openeo_connection()
print("Conectado a:", con.root_url)
print("Usuario autenticado correctamente.")

### Ejercicio 2 — Descarga de datos raster por lago y fecha

Se descargan únicamente:
- **B03, B04, B08** (vía openEO) para calcular NDVI y NDWI localmente.
- El resultado del **script oficial de cianobacteria** de Sentinel Hub (vía Process API) para cada fecha — no se descargan bandas completas de más para este índice.

Se usan exclusivamente las fechas oficiales del enunciado (11 por lago) para minimizar tiempo de descarga y asegurar reproducibilidad entre grupos.

In [ ]:
from tqdm.auto import tqdm

rutas_bandas = {}
for lago, info in LAGOS.items():
    rutas_bandas[lago] = {}
    for fecha in tqdm(info["fechas"], desc=f"Descargando bandas {lago}"):
        try:
            ruta = sentinel_api.descargar_bandas(con, lago, fecha)
            rutas_bandas[lago][fecha] = ruta
        except Exception as e:
            print(f"[{lago} - {fecha}] ERROR descargando bandas: {e}")

rutas_bandas

In [ ]:
rutas_cyano = {}
for lago, info in LAGOS.items():
    rutas_cyano[lago] = {}
    for fecha in tqdm(info["fechas"], desc=f"Descargando cianobacteria {lago}"):
        try:
            ruta = sentinel_api.descargar_cyano(lago, fecha)
            rutas_cyano[lago][fecha] = ruta
        except Exception as e:
            print(f"[{lago} - {fecha}] ERROR descargando cyano: {e}")

rutas_cyano

### Verificación rápida

Confirmamos que los archivos se descargaron en `data/raw/<lago>/` y que tienen las bandas/dimensiones esperadas antes de pasar al Notebook 2 (índices y análisis temporal).

In [ ]:
import rasterio

for lago in LAGOS:
    for fecha, ruta in rutas_bandas.get(lago, {}).items():
        with rasterio.open(ruta) as src:
            print(lago, fecha, "bandas:", src.count, "tamaño:", src.width, "x", src.height)